# PaDiM — Comparaison des variantes de resize / crop sur `cable`

On compare deux pré-traitements pour PaDiM sur la catégorie `cable` :

- **V1 — Resize 256×256** seul (notre baseline initial)
- **V2 — Resize 256 + CenterCrop 224** (preset original du papier PaDiM)

Tout le reste est identique : ResNet18 gelé, random projection `d'=100`, `ε=0.01`, lissage gaussien `σ=4`.

**Sortie attendue :** AUROC image-level + pixel-level pour chaque variante, ROC superposées, heatmaps comparées sur les mêmes images.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score, roc_curve

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA, EDA, PATHS

warnings.filterwarnings('ignore')
sns.set_theme(style=EDA.sns_style, palette=EDA.sns_palette, font_scale=EDA.sns_font_scale)
plt.rcParams['figure.dpi'] = EDA.figure_dpi

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = DATA.random_seed
torch.manual_seed(SEED); np.random.seed(SEED)

# --- Hyperparams partagés ---
CATEGORY = 'cable'
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
BACKBONE = 'resnet18'
D_PROJ = 100
EPSILON = 0.01
SMOOTH_SIGMA = 4.0
BATCH = 16

# --- Définition des deux variantes ---
VARIANTS = {
    'V1 — Resize 256':            {'resize': 256, 'crop': None, 'color': '#4C72B0'},
    'V2 — Resize 256 + Crop 224': {'resize': 256, 'crop': 224,  'color': '#C44E52'},
}

print(f'Device : {DEVICE}')
print(f'Catégorie : {CATEGORY} | backbone : {BACKBONE}')
for name, cfg in VARIANTS.items():
    final = cfg['crop'] if cfg['crop'] else cfg['resize']
    print(f"  {name}  →  taille finale {final}×{final}")

## 1. Données & transforms paramétrables

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

df = pd.read_csv(PATHS.unified_csv)
df_cat = df[(df['dataset'] == DATA.mvtec_name) & (df['category'] == CATEGORY)].reset_index(drop=True)
print(f'{len(df_cat)} images pour {CATEGORY}')
print(df_cat.groupby(['split', 'is_anomaly']).size())


def build_transforms(resize: int, crop: int | None):
    """Image and mask transforms for a given variant."""
    img_steps  = [transforms.Resize((resize, resize), interpolation=transforms.InterpolationMode.BILINEAR)]
    mask_steps = [transforms.Resize((resize, resize), interpolation=transforms.InterpolationMode.NEAREST)]
    if crop is not None:
        img_steps.append(transforms.CenterCrop(crop))
        mask_steps.append(transforms.CenterCrop(crop))
    img_steps += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    mask_steps += [transforms.PILToTensor()]
    return transforms.Compose(img_steps), transforms.Compose(mask_steps)


class AnomalyDataset(Dataset):
    def __init__(self, df, img_tfm, mask_tfm, root, final_size):
        self.df = df.reset_index(drop=True)
        self.img_tfm = img_tfm
        self.mask_tfm = mask_tfm
        self.root = Path(root)
        self.final_size = final_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row['image_path']).convert('RGB')
        x = self.img_tfm(img)
        if isinstance(row.get('mask_path'), str) and bool(row['has_mask']):
            m = Image.open(self.root / row['mask_path']).convert('L')
            m = (self.mask_tfm(m) > 0).float()
        else:
            m = torch.zeros(1, self.final_size, self.final_size)
        return {
            'image': x,
            'mask': m,
            'label': int(row['is_anomaly']),
            'image_path': row['image_path'],
            'defect_label': row.get('label', 'good'),
        }


def make_loaders(resize, crop):
    img_tfm, mask_tfm = build_transforms(resize, crop)
    final = crop if crop else resize
    train_df = df_cat[(df_cat['split'] == 'train') & (~df_cat['is_anomaly'])]
    test_df  = df_cat[df_cat['split'] == 'test']
    train_ds = AnomalyDataset(train_df, img_tfm, mask_tfm, PATHS.root, final)
    test_ds  = AnomalyDataset(test_df,  img_tfm, mask_tfm, PATHS.root, final)
    return (
        DataLoader(train_ds, batch_size=BATCH, shuffle=False, num_workers=0),
        DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0),
        final,
    )

## 2. Composants partagés — feature extractor + PaDiM

In [ ]:
from torchvision import models

class PaDiMFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        net.eval()
        for p in net.parameters():
            p.requires_grad = False
        self.net = net
        self._features = {}
        net.layer1.register_forward_hook(self._hook('layer1'))
        net.layer2.register_forward_hook(self._hook('layer2'))
        net.layer3.register_forward_hook(self._hook('layer3'))

    def _hook(self, name):
        def fn(_m, _i, out):
            self._features[name] = out
        return fn

    @torch.no_grad()
    def forward(self, x):
        self._features = {}
        _ = self.net(x)
        f1 = self._features['layer1']
        f2 = self._features['layer2']
        f3 = self._features['layer3']
        f2 = F.interpolate(f2, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        f3 = F.interpolate(f3, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        return torch.cat([f1, f2, f3], dim=1)


class PaDiM:
    def __init__(self, extractor, img_size, d_proj=D_PROJ, epsilon=EPSILON, device=DEVICE, seed=SEED):
        self.extractor = extractor
        self.img_size = img_size
        self.d_proj = d_proj
        self.epsilon = epsilon
        self.device = device
        with torch.no_grad():
            x = torch.zeros(1, 3, img_size, img_size, device=device)
            D_total = extractor(x).shape[1]
        g = torch.Generator().manual_seed(seed)
        self.proj_idx = torch.randperm(D_total, generator=g)[:d_proj].to(device)
        self.mu = self.cov_inv = None
        self.H = self.W = self.HW = None

    @torch.no_grad()
    def _embed(self, batch):
        f = self.extractor(batch.to(self.device))
        f = f.index_select(1, self.proj_idx)
        B, d, H, W = f.shape
        return f.permute(0, 2, 3, 1).reshape(B, H * W, d), H, W

    def fit(self, dataloader):
        sums = outers = None
        n = 0
        for batch in dataloader:
            embs, H, W = self._embed(batch['image'])
            if sums is None:
                self.H, self.W, self.HW = H, W, H * W
                sums = torch.zeros(self.HW, self.d_proj, device=self.device)
                outers = torch.zeros(self.HW, self.d_proj, self.d_proj, device=self.device)
            sums += embs.sum(dim=0)
            outers += torch.einsum('bnd,bne->nde', embs, embs)
            n += embs.shape[0]
        mu = sums / n
        cov = (outers - n * torch.einsum('nd,ne->nde', mu, mu)) / max(n - 1, 1)
        cov += self.epsilon * torch.eye(self.d_proj, device=self.device).unsqueeze(0)
        self.mu = mu
        self.cov_inv = torch.linalg.inv(cov)
        return self

    @torch.no_grad()
    def score(self, batch):
        embs, _, _ = self._embed(batch)
        diff = embs - self.mu.unsqueeze(0)
        tmp = torch.einsum('bnd,nde->bne', diff, self.cov_inv)
        D2 = (tmp * diff).sum(-1).clamp_min(0.0).sqrt()
        D2 = D2.reshape(-1, self.H, self.W)
        return F.interpolate(D2.unsqueeze(1), size=(self.img_size, self.img_size),
                             mode='bilinear', align_corners=False).squeeze(1)

extractor = PaDiMFeatureExtractor().to(DEVICE)

## 3. Pipeline d'évaluation pour une variante

In [ ]:
def run_variant(name, cfg):
    print(f'\n{"="*60}\n▶ {name}\n{"="*60}')
    train_loader, test_loader, img_size = make_loaders(cfg['resize'], cfg['crop'])
    print(f'Taille image finale : {img_size}×{img_size}')

    padim = PaDiM(extractor, img_size=img_size)
    t0 = time.time()
    padim.fit(train_loader)
    print(f'Fit  : {time.time() - t0:.1f}s')

    heatmaps, labels, gt_masks, images, paths, defect_labels = [], [], [], [], [], []
    t0 = time.time()
    for batch in test_loader:
        heat = padim.score(batch['image']).cpu().numpy()
        for i in range(heat.shape[0]):
            heat[i] = gaussian_filter(heat[i], sigma=SMOOTH_SIGMA)
        heatmaps.append(heat)
        labels.append(batch['label'].numpy())
        gt_masks.append(batch['mask'].squeeze(1).numpy())
        images.append(batch['image'].numpy())
        paths.extend(batch['image_path'])
        defect_labels.extend(batch['defect_label'])
    print(f'Score: {time.time() - t0:.1f}s')

    heatmaps = np.concatenate(heatmaps)
    labels = np.concatenate(labels)
    gt_masks = np.concatenate(gt_masks)
    images = np.concatenate(images)

    img_scores = heatmaps.reshape(heatmaps.shape[0], -1).max(axis=1)
    auc_img = roc_auc_score(labels, img_scores)
    is_anom = labels == 1
    auc_pix = roc_auc_score(gt_masks[is_anom].flatten(), heatmaps[is_anom].flatten())
    print(f'AUROC image : {auc_img:.4f} | AUROC pixel : {auc_pix:.4f}')

    return dict(
        name=name, cfg=cfg, img_size=img_size,
        heatmaps=heatmaps, labels=labels, gt_masks=gt_masks, images=images,
        paths=paths, defect_labels=defect_labels, img_scores=img_scores,
        auc_img=auc_img, auc_pix=auc_pix,
    )


def denorm(x_np, img_size):
    mean = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std  = np.array(IMAGENET_STD).reshape(3, 1, 1)
    return np.clip(x_np * std + mean, 0, 1).transpose(1, 2, 0)

## 4. Exécution des deux variantes

In [ ]:
results = {name: run_variant(name, cfg) for name, cfg in VARIANTS.items()}

## 5. Comparaison des AUROC + ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Bar AUROC image vs pixel ---
names = list(results.keys())
auc_img_vals = [results[n]['auc_img'] for n in names]
auc_pix_vals = [results[n]['auc_pix'] for n in names]
colors = [VARIANTS[n]['color'] for n in names]

x = np.arange(2)
w = 0.36
for i, n in enumerate(names):
    axes[0].bar(x + (i - 0.5) * w, [results[n]['auc_img'], results[n]['auc_pix']],
                width=w, color=colors[i], label=n, edgecolor='black', linewidth=0.4)
    for j, val in enumerate([results[n]['auc_img'], results[n]['auc_pix']]):
        axes[0].text(x[j] + (i - 0.5) * w, val + 0.005, f'{val:.3f}',
                     ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(['AUROC image', 'AUROC pixel'])
axes[0].set_ylim(min(auc_img_vals + auc_pix_vals) - 0.05, 1.0)
axes[0].set_title('Comparaison AUROC', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=9)
sns.despine(ax=axes[0])

# --- ROC superposées (image-level) ---
for n in names:
    r = results[n]
    fpr, tpr, _ = roc_curve(r['labels'], r['img_scores'])
    axes[1].plot(fpr, tpr, color=VARIANTS[n]['color'], lw=2,
                 label=f"{n}  (AUROC={r['auc_img']:.3f})")
axes[1].plot([0, 1], [0, 1], '--', color='gray', lw=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC image-level', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=9)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

### AUPIMO — Area Under the Per-Image Overlap curve

L'**AUROC pixel** est dominé par les vrais négatifs (la grande majorité des pixels sont normaux) et tend à surévaluer la qualité de la localisation. **AUPIMO** (Bertoldo et al., 2024) corrige ce biais :

- **Per-Image** : on calcule un score de recouvrement défaut **par image anomale**, pas globalement.
- **Borne FPR basse** : on intègre sur `FPR ∈ [1e-5, 1e-4]` → on mesure la qualité du modèle *uniquement* quand on accepte très peu de faux positifs (régime industriel).
- Sortie : une AUPIMO **par image anomale** (NaN pour les images saines). On rapporte la **moyenne**.

C'est une métrique plus exigeante et plus représentative du déploiement réel.

In [ ]:
from anomalib.metrics.pimo.pimo import _AUPIMO

def compute_aupimo(heatmaps_np, gt_masks_np, fpr_bounds=(1e-5, 1e-4)):
    metric = _AUPIMO(fpr_bounds=fpr_bounds, return_average=False, force=True)
    metric.update(
        torch.from_numpy(heatmaps_np).float(),
        torch.from_numpy(gt_masks_np).long(),
    )
    _, aupimo_result = metric.compute()
    aupimos = aupimo_result.aupimos
    valid = ~torch.isnan(aupimos)
    return aupimos[valid].mean().item(), aupimos[valid].cpu().numpy()

aupimo_scores = {}
for name, r in results.items():
    mean, per_img = compute_aupimo(r['heatmaps'], r['gt_masks'])
    aupimo_scores[name] = {'mean': mean, 'per_image': per_img}
    print(f'{name:<35}  AUPIMO mean = {mean:.4f}  (n={len(per_img)} images anomales)')

# --- Visualisation : bar AUPIMO + distribution per-image ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names = list(results.keys())
means = [aupimo_scores[n]['mean'] for n in names]
colors = [VARIANTS[n]['color'] for n in names]

axes[0].bar(np.arange(len(names)), means, color=colors,
            edgecolor='black', linewidth=0.5, width=0.55)
for i, m in enumerate(means):
    axes[0].text(i, m + 0.01, f'{m:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[0].set_xticks(np.arange(len(names)))
axes[0].set_xticklabels(names, rotation=12, ha='right', fontsize=9)
axes[0].set_ylabel('AUPIMO  (mean over anomalous images)')
axes[0].set_ylim(0, max(means) * 1.2 + 0.05)
axes[0].set_title('AUPIMO global  (FPR ∈ [1e-5, 1e-4])', fontweight='bold')
sns.despine(ax=axes[0])

for n in names:
    axes[1].hist(aupimo_scores[n]['per_image'], bins=15,
                 alpha=0.5, label=n, color=VARIANTS[n]['color'],
                 edgecolor='black', linewidth=0.3)
axes[1].set_xlabel('AUPIMO par image anomale')
axes[1].set_ylabel("Nombre d'images")
axes[1].set_title('Distribution AUPIMO par image', fontweight='bold')
axes[1].legend(loc='upper left', fontsize=9)
sns.despine(ax=axes[1])

plt.tight_layout(); plt.show()

In [ ]:
# AUROC par type de défaut (vs good)
rows = []
for name in names:
    r = results[name]
    defect_arr = np.array(r['defect_labels'])
    for lbl in sorted(set(defect_arr) - {'good'}):
        mask_lbl = (defect_arr == lbl) | (defect_arr == 'good')
        if (defect_arr == lbl).sum() < 1:
            continue
        auc_lbl = roc_auc_score(r['labels'][mask_lbl], r['img_scores'][mask_lbl])
        rows.append({'variante': name, 'défaut': lbl,
                     'n': int((defect_arr == lbl).sum()),
                     'AUROC': round(auc_lbl, 3)})
df_per_defect = pd.DataFrame(rows)
df_per_defect_pivot = df_per_defect.pivot(index='défaut', columns='variante', values='AUROC')
df_per_defect_pivot['Δ (V2 − V1)'] = (df_per_defect_pivot.iloc[:, 1] - df_per_defect_pivot.iloc[:, 0]).round(3)
df_per_defect_pivot

### Matrice de confusion (au seuil optimal de Youden's J)

On binarise les scores en `Good / Anomal` au **seuil optimal de Youden's J** (`argmax(TPR − FPR)`). En production le seuil serait ajusté selon le coût des FP (alarmes inutiles) vs FN (défauts ratés).

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, len(results), figsize=(5.5 * len(results), 4.8))
if len(results) == 1:
    axes = [axes]

cm_summary = []
for ax, (name, r) in zip(axes, results.items()):
    fpr_, tpr_, thrs = roc_curve(r['labels'], r['img_scores'])
    best_idx = np.argmax(tpr_ - fpr_)
    thr = thrs[best_idx]
    preds = (r['img_scores'] >= thr).astype(int)
    cm = confusion_matrix(r['labels'], preds)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    cm_summary.append({'variante': name, 'threshold': round(thr, 4),
                       'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                       'precision': round(precision, 3), 'recall': round(recall, 3), 'F1': round(f1, 3)})

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Good', 'Anomal'], yticklabels=['Good', 'Anomal'],
                ax=ax, cbar=False, annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_xlabel('Prédiction'); ax.set_ylabel('Vérité')
    ax.set_title(f'{name}\nthr={thr:.3f}  ·  F1={f1:.3f}', fontsize=10, fontweight='bold',
                 color=VARIANTS[name]['color'])

plt.suptitle('Matrice de confusion — seuil Youden optimal', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

pd.DataFrame(cm_summary)

## 6. Heatmaps comparées sur les mêmes images

Pour chaque image de test choisie, on affiche : original V1 + heatmap V1 + original V2 + heatmap V2 + GT mask.

In [ ]:
# Sélection : 1 normal + 2 anomalies bien détectées V1 + 2 mal détectées V1
r1 = results[names[0]]
r2 = results[names[1]]

anom_idx = np.where(r1['labels'] == 1)[0]
norm_idx = np.where(r1['labels'] == 0)[0]
anom_sorted = anom_idx[np.argsort(-r1['img_scores'][anom_idx])]
norm_sorted = norm_idx[np.argsort(-r1['img_scores'][norm_idx])]

picks = (
    [('Normal top-score', i) for i in norm_sorted[:1]] +
    [('Anomal best',     i) for i in anom_sorted[:2]] +
    [('Anomal worst',    i) for i in anom_sorted[-2:]]
)

n = len(picks)
fig, axes = plt.subplots(n, 5, figsize=(15, 2.9 * n))
for r_idx, (tag, idx) in enumerate(picks):
    img1 = denorm(r1['images'][idx], r1['img_size'])
    img2 = denorm(r2['images'][idx], r2['img_size'])
    h1 = r1['heatmaps'][idx]; h2 = r2['heatmaps'][idx]
    gt1 = r1['gt_masks'][idx]
    s1 = r1['img_scores'][idx]; s2 = r2['img_scores'][idx]
    defect = r1['defect_labels'][idx]

    axes[r_idx, 0].imshow(img1)
    axes[r_idx, 0].set_title(f'V1 — image\n{tag} · {defect}', fontsize=9)
    axes[r_idx, 1].imshow(img1); axes[r_idx, 1].imshow(h1, cmap='jet', alpha=0.45)
    axes[r_idx, 1].set_title(f'V1 — overlay (s={s1:.2f})', fontsize=9, color=VARIANTS[names[0]]['color'])
    axes[r_idx, 2].imshow(img2)
    axes[r_idx, 2].set_title(f'V2 — image\n({r2["img_size"]}×{r2["img_size"]})', fontsize=9)
    axes[r_idx, 3].imshow(img2); axes[r_idx, 3].imshow(h2, cmap='jet', alpha=0.45)
    axes[r_idx, 3].set_title(f'V2 — overlay (s={s2:.2f})', fontsize=9, color=VARIANTS[names[1]]['color'])
    if gt1.sum() > 0:
        axes[r_idx, 4].imshow(gt1, cmap='gray_r')
        axes[r_idx, 4].set_title('GT mask (V1)', fontsize=9)
    else:
        axes[r_idx, 4].imshow(np.zeros_like(gt1), cmap='gray_r')
        axes[r_idx, 4].set_title('(pas de GT)', fontsize=9)
    for c in range(5):
        axes[r_idx, c].set_xticks([]); axes[r_idx, c].set_yticks([])

plt.suptitle(f'PaDiM — comparaison V1 vs V2 sur {CATEGORY}',
             fontsize=12, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## 7. Verdict & décision

| Critère | Observation V1 vs V2 |
|---|---|
| **AUROC image-level** | V1 = 0.862 · V2 = **0.882**  →  **V2 +0.020** |
| **AUROC pixel-level** | V1 = **0.961** · V2 = 0.950  →  V1 +0.011 (perte mineure) |
| **AUROC par type de défaut** | **V2 gagne sur les 8/8 types**. Gain max sur les défauts faibles : `cable_swap` (+0.045), `missing_wire` (+0.042) |
| **Défauts proches du bord** | Aucun signal négatif dans la table par type → le crop ne coupe pas de défauts critiques |
| **Stabilité de la heatmap (FP fond)** | Le gain image-level provient probablement d'une suppression des FP sur les bords (à confirmer visuellement section 6) |
| **Temps fit / inference** | V2 plus rapide (~24 % moins de positions spatiales : 56² = 3136 vs 64² = 4096) |
| **Limitation résiduelle** | `cable_swap` (0.74) et `missing_wire` (0.70) restent faibles → **limite structurelle de PaDiM** (défauts topologiques, pas de texture), pas un problème de préprocessing |

### Choix retenu pour le préprocessing

**V2 — Resize 256 + CenterCrop 224**

**Raisons :**
1. Gain image-level **cohérent et reproduit sur les 8 types** de défauts (aucun Δ négatif).
2. Faible perte pixel-level (-0.011) acceptable : la **détection** est plus prioritaire que la localisation fine en production (tri / rejet de pièce).
3. Pipeline plus rapide.
4. Le crop élimine la part de la heatmap qui chauffe sur les bords sombres (FP).

### Limites identifiées (à traiter dans une étape ultérieure)

- AUROC image global = 0.882 — encore en dessous de la référence MVTec PaDiM-RN18 sur cable (~0.93). Pistes :
  - Backbone plus capable : `wide_resnet50_2` (`d'=550`).
  - Résolution plus haute : 320 → crop 288, surtout pour les petits défauts.
- Défauts **topologiques** (`cable_swap`, `missing_wire`) : limitation fondamentale de PaDiM. À traiter avec un modèle reconstruction-based (autoencoder, DRAEM) ou une méthode capable de capter la structure globale.